In [1]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

# Reads the repo's own CSV. The Colab original began with files.upload(); that
# step is unnecessary outside Colab and is what made the notebook non-portable.
df = pd.read_csv('features_2000.csv')
print(len(df), 'raw')

df = df.drop_duplicates(subset='kepoi_name', keep='last')
print(len(df), 'after dedup')      # expect 2289

ok = df[df['error'].isna()].copy()
print(len(ok), 'successful rows')

# LIGHT-CURVE FEATURES ONLY.
# duration_over_period was removed and koi_impact was never added: both come from
# the catalogue's transit-model fit, and including either makes the project's
# central claim untestable. The ablation that settled this is in
# ablate_features.py; the reasoning is CLAUDE.md section 11.
FEATURES = ['depth', 'width', 'asymmetry', 'snr',
            'odd_even_diff', 'secondary_depth']

X = ok[FEATURES]
y = (ok['disposition'] == 'CONFIRMED').astype(int)
groups = ok['kepid']

2295 raw
2289 after dedup
2224 successful rows


In [2]:
outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(outer.split(X, y, groups))

inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
a, b = next(inner.split(X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]))
tr, va = train_idx[a], train_idx[b]

print(f"train {len(train_idx)}  (inner fit {len(tr)} / validation {len(va)})")
print(f"test  {len(test_idx)}")
print("star-disjoint:", not (set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

train 1790  (inner fit 1352 / validation 438)
test  434
star-disjoint: True


In [3]:
# Untuned, matching the final model below. The threshold is chosen HERE, on a
# validation split carved out of the training stars -- the test set is not touched
# until the last cell. Grid search in the next cell; it was not adopted.

m = HistGradientBoostingClassifier(random_state=42).fit(X.iloc[tr], y.iloc[tr])
pv = m.predict_proba(X.iloc[va])[:, 1]

for t in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    p = (pv >= t).astype(int)
    tp = ((p == 1) & (y.iloc[va] == 1)).sum()
    fp = ((p == 1) & (y.iloc[va] == 0)).sum()
    fn = ((p == 0) & (y.iloc[va] == 1)).sum()
    print(f"{t}: recall {tp/(tp+fn):.2f}, precision {tp/(tp+fp):.2f}")

0.1: recall 0.94, precision 0.65
0.15: recall 0.91, precision 0.67
0.2: recall 0.89, precision 0.69
0.25: recall 0.89, precision 0.71
0.3: recall 0.86, precision 0.71
0.4: recall 0.81, precision 0.74
0.5: recall 0.79, precision 0.75


In [4]:
# Previously commented out, so its result could not be checked. Now live, because
# an unverifiable claim in a notebook is worse than 50 seconds of compute.
from sklearn.model_selection import GridSearchCV, GroupKFold
import numpy as np

grid = {
    'learning_rate': [0.05, 0.1],
    'max_leaf_nodes': [15, 31],
    'min_samples_leaf': [20, 40],
    'l2_regularization': [0.0, 1.0],
    'max_iter': [100, 300],
}

search = GridSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    grid,
    scoring='average_precision',
    cv=GroupKFold(n_splits=5),
    n_jobs=2,
)
search.fit(X.iloc[train_idx], y.iloc[train_idx], groups=groups.iloc[train_idx])

default_score = search.cv_results_['mean_test_score'][
    np.where((search.cv_results_['param_learning_rate'] == 0.1) &
             (search.cv_results_['param_max_leaf_nodes'] == 31) &
             (search.cv_results_['param_min_samples_leaf'] == 20) &
             (search.cv_results_['param_l2_regularization'] == 0.0) &
             (search.cv_results_['param_max_iter'] == 100))[0][0]
]

r = search.cv_results_
best_i = search.best_index_
default_i = int(np.where((r['param_learning_rate'] == 0.1) &
                         (r['param_max_leaf_nodes'] == 31) &
                         (r['param_min_samples_leaf'] == 20) &
                         (r['param_l2_regularization'] == 0.0) &
                         (r['param_max_iter'] == 100))[0][0])

gap = r['mean_test_score'][best_i] - r['mean_test_score'][default_i]
pooled = np.sqrt(r['std_test_score'][best_i]**2 + r['std_test_score'][default_i]**2)

print(f"{len(r['params'])} configurations")
print(f"best params:   {search.best_params_}")
print(f"best CV:       {r['mean_test_score'][best_i]:.4f} ± {r['std_test_score'][best_i]:.4f}")
print(f"default CV:    {r['mean_test_score'][default_i]:.4f} ± {r['std_test_score'][default_i]:.4f}")
print(f"gap:           {gap:+.4f}   pooled fold std {pooled:.4f}   ratio {gap/pooled:.2f}")
print()
print("Not adopted. The gap is a quarter of the fold-to-fold spread, so it is")
print("not distinguishable from which stars happened to land in which fold.")

32 configurations
best params:   {'l2_regularization': 1.0, 'learning_rate': 0.05, 'max_iter': 100, 'max_leaf_nodes': 15, 'min_samples_leaf': 20}
best CV:       0.8256 ± 0.0622
default CV:    0.8034 ± 0.0544
gap:           +0.0223   pooled fold std 0.0826   ratio 0.27

Not adopted. The gap is a quarter of the fold-to-fold spread, so it is
not distinguishable from which stars happened to land in which fold.


In [5]:
THRESHOLD = 0.25   # <- chosen in the threshold sweep two cells above

final = HistGradientBoostingClassifier(random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
probs = final.predict_proba(X.iloc[test_idx])[:, 1]
preds = (probs >= THRESHOLD).astype(int)

print(classification_report(y.iloc[test_idx], preds,
                            target_names=['FALSE POSITIVE', 'CONFIRMED']))
print(confusion_matrix(y.iloc[test_idx], preds))

imp = permutation_importance(final, X.iloc[test_idx], y.iloc[test_idx],
                             n_repeats=30, random_state=42, scoring='accuracy')
for n, mn, sd in sorted(zip(FEATURES, imp.importances_mean, imp.importances_std),
                        key=lambda t: -t[1]):
    print(f"{n:22s} {mn:+.4f} ± {sd:.4f}")

                precision    recall  f1-score   support

FALSE POSITIVE       0.89      0.74      0.81       301
     CONFIRMED       0.58      0.80      0.67       133

      accuracy                           0.76       434
     macro avg       0.73      0.77      0.74       434
  weighted avg       0.80      0.76      0.77       434

[[223  78]
 [ 27 106]]
depth                  +0.0627 ± 0.0154
secondary_depth        +0.0541 ± 0.0130
snr                    +0.0300 ± 0.0142
odd_even_diff          +0.0190 ± 0.0122
width                  +0.0185 ± 0.0156
asymmetry              +0.0043 ± 0.0129


In [6]:
# Star-grouped splitting means the class mix differs between splits. Worth
# printing, because a baseline quoted against the wrong split is misleading --
# see CLAUDE.md 11.6.
print(f"CONFIRMED rate, whole dataset: {y.mean():.4f}")
print(f"CONFIRMED rate, validation:    {y.iloc[va].mean():.4f}")
print(f"CONFIRMED rate, test:          {y.iloc[test_idx].mean():.4f}")
print(f"majority-class accuracy on test: {1 - y.iloc[test_idx].mean():.4f}")

CONFIRMED rate, whole dataset: 0.3660
CONFIRMED rate, validation:    0.3630
CONFIRMED rate, test:          0.3065
majority-class accuracy on test: 0.6935


In [7]:
import joblib

joblib.dump({
    'model': final,
    'features': FEATURES,
    'threshold': THRESHOLD,
}, 'transitcheck_model.joblib')

print('saved transitcheck_model.joblib')
print('  features :', FEATURES)
print('  threshold:', THRESHOLD)

saved transitcheck_model.joblib
  features : ['depth', 'width', 'asymmetry', 'snr', 'odd_even_diff', 'secondary_depth']
  threshold: 0.25
